# 01 · Carga y auditoría de los 24 archivos originales

Lee los archivos `MC4_PSQ *.xlsx` **sin ediciones manuales** y audita: esquema único,
cobertura temporal, duplicados, huecos/traslapes entre quincenas y columnas sin
información. Los datos se toman de `TERNIUM_DATA_DIR` (ver README).

In [1]:
import sys
sys.path.insert(0, "..")  # para importar src/

In [2]:
# Lee los 24 archivos originales y audita esquema, cobertura, duplicados y huecos.
import pandas as pd

from src.carga import list_files, file_period, load_all, SHEET_NAME
from src.depuracion import columnas_sin_variacion
from src.poblacion import COLUMNAS_SKP

## 1. Inventario de archivos y periodos declarados

In [3]:
files = list_files()
periodos = pd.DataFrame(
    [{"archivo": f.name, "inicio": file_period(f)[0], "fin": file_period(f)[1]} for f in files]
)
print(f"{len(files)} archivos")
print(periodos.head(3).to_string(index=False))

24 archivos
                         archivo     inicio                 fin
  MC4_PSQ Enero 1 a 15 2025.xlsx 2025-01-01 2025-01-15 23:59:59
 MC4_PSQ Enero 16 a 31 2025.xlsx 2025-01-16 2025-01-31 23:59:59
MC4_PSQ Febrero 1 a 15 2025.xlsx 2025-02-01 2025-02-15 23:59:59


## 2. Esquema: ¿los 24 archivos tienen las mismas columnas?

In [4]:
esquemas = {}
for f in files:
    cols = tuple(pd.read_excel(f, sheet_name=SHEET_NAME, engine="calamine", nrows=1).columns.str.strip())
    esquemas.setdefault(cols, []).append(f.name)
print(f"\nEsquemas distintos: {len(esquemas)} | columnas: {len(next(iter(esquemas)))}")


Esquemas distintos: 1 | columnas: 280


## 3. Carga completa

In [5]:
df = load_all()
print(f"\nRollos: {len(df):,} | columnas (con 2 de metadatos): {df.shape[1]}")
print("Rollos por quincena: min", df.groupby("archivo").size().min(),
      "| max", df.groupby("archivo").size().max())


Rollos: 149,734 | columnas (con 2 de metadatos): 282
Rollos por quincena: min 4038 | max 7405


## 4. Cobertura temporal, duplicados y huecos

In [6]:
f0, f1 = df["Fecha Inicio"].min(), df["Fecha Inicio"].max()
print(f"\nCobertura: {f0} → {f1}")
print("Duplicados de Material Salida:", df["Material Salida"].duplicated().sum())
print("Duplicados de Material Entrada (planchón):", df["Material Entrada"].duplicated().sum())
orden = df["Fecha Inicio"].sort_values()
print("Hueco máximo entre rollos consecutivos (h):", round(orden.diff().max().total_seconds() / 3600, 2))

# ¿Traslapes o huecos ENTRE archivos? Último rollo de cada quincena vs. primero de la siguiente
tiempos = df.groupby("archivo")["Fecha Inicio"].agg(["min", "max"]).sort_values("min")
frontera = (tiempos["min"].shift(-1) - tiempos["max"]).dropna()
print("Archivos cuyo periodo se traslapa con el siguiente:",
      int((frontera < pd.Timedelta(0)).sum()))
print("Hueco máximo en la frontera entre quincenas (h):",
      round(frontera.max().total_seconds() / 3600, 2),
      "→ el hueco de 90 h es un paro DENTRO de una quincena, no un archivo faltante")


Cobertura: 2024-12-31 18:58:08 → 2025-12-31 18:57:23
Duplicados de Material Salida: 0
Duplicados de Material Entrada (planchón): 0
Hueco máximo entre rollos consecutivos (h): 90.04
Archivos cuyo periodo se traslapa con el siguiente: 0
Hueco máximo en la frontera entre quincenas (h): 0.11 → el hueco de 90 h es un paro DENTRO de una quincena, no un archivo faltante


## 5. Columnas sin información

In [7]:
vacias, constantes = columnas_sin_variacion(df)
print(f"\nColumnas 100 % vacías: {len(vacias)} | constantes: {len(constantes)}")
print("Columnas del Skinpass (no vacías → rollos):")
print(df[COLUMNAS_SKP].notna().sum().to_string())


Columnas 100 % vacías: 42 | constantes: 38
Columnas del Skinpass (no vacías → rollos):
Línea Adelantada SKP    0
% Elongación Mínimo     0
% Elongación Máximo     0
Espesor Mínimo SKP      0
Espesor Máximo SKP      0
